# DSA-210 Project – Sleep & Weather Analysis

This notebook performs exploratory data analysis (EDA) and hypothesis testing on:
- **sleep_data.csv** (personal sleep data)
- **external_data_pendik.csv** (Pendik / Istanbul daily weather data)

Files should be in the **same folder** as this notebook.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats

# Display all columns
pd.set_option('display.max_columns', None)

# Load data (adjust paths if you put them in a subfolder like 'data/')
sleep = pd.read_csv('sleep_data.csv', parse_dates=['date'])
external = pd.read_csv('external_data_pendik.csv', parse_dates=['date'])

sleep.head(), external.head()

## 1. Merge Sleep & Weather Data

In [ ]:
# Merge on date
df = pd.merge(sleep, external, on='date', how='inner')

print('Shape:', df.shape)
df.head()

## 2. Basic Summary Statistics

In [ ]:
print('--- INFO ---')
print(df.info())

print('\n--- DESCRIBE (numeric columns) ---')
df.describe(numeric_only=True)

## 3. Missing Values

In [ ]:
print(df.isna().sum())

# For simplicity, drop rows with any missing values (if any)
df = df.dropna().reset_index(drop=True)
print('\nAfter dropna, shape:', df.shape)

## 4. Feature Engineering – Weekday / Weekend

In [ ]:
df['weekday'] = df['date'].dt.weekday  # Monday = 0, Sunday = 6
df['is_weekend'] = df['weekday'] >= 5

df[['date', 'weekday', 'is_weekend']].head()

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Histogram of total sleep hours
plt.figure()
plt.hist(df['total_sleep_hours'], bins=10)
plt.title('Distribution of Total Sleep Hours')
plt.xlabel('Total Sleep Hours')
plt.ylabel('Frequency')
plt.show()

# Histogram of sleep score
plt.figure()
plt.hist(df['sleep_score'], bins=10)
plt.title('Distribution of Sleep Score')
plt.xlabel('Sleep Score')
plt.ylabel('Frequency')
plt.show()

# Scatter-like relationship: average temp vs total sleep (using simple plot)
plt.figure()
plt.scatter(df['avg_temp'], df['total_sleep_hours'])
plt.title('Average Temperature vs Total Sleep Hours')
plt.xlabel('Average Temperature (°C)')
plt.ylabel('Total Sleep Hours')
plt.show()


## 6. Hypothesis Test 1 – Weekday vs Weekend Sleep
**H0 (Null Hypothesis):** There is **no difference** in mean total sleep hours between weekdays and weekends.

**H1 (Alternative Hypothesis):** Mean total sleep hours on weekends is **different** from weekdays.


In [ ]:
weekdays = df.loc[~df['is_weekend'], 'total_sleep_hours']
weekends = df.loc[df['is_weekend'], 'total_sleep_hours']

t_stat, p_value = stats.ttest_ind(weekdays, weekends, equal_var=False)
print('T-statistic:', t_stat)
print('p-value:', p_value)

if p_value < 0.05:
    print('Result: Reject H0 at 5% significance level. There is a significant difference.')
else:
    print('Result: Fail to reject H0 at 5% significance level. No significant difference found.')

## 7. Hypothesis Test 2 – Temperature vs Deep Sleep
**H0:** There is **no difference** in mean deep sleep hours between cooler and warmer days.

**H1:** Mean deep sleep hours is **different** between cooler and warmer days.

We split days into two groups by the median of `avg_temp`.

In [ ]:
median_temp = df['avg_temp'].median()

cool_days = df.loc[df['avg_temp'] <= median_temp, 'deep_sleep_hours']
warm_days = df.loc[df['avg_temp'] > median_temp, 'deep_sleep_hours']

t_stat2, p_value2 = stats.ttest_ind(cool_days, warm_days, equal_var=False)
print('T-statistic:', t_stat2)
print('p-value:', p_value2)

if p_value2 < 0.05:
    print('Result: Reject H0 at 5% significance level. There is a significant difference.')
else:
    print('Result: Fail to reject H0 at 5% significance level. No significant difference found.')

## 8. Hypothesis Test 3 – Rain vs No-Rain Sleep Score
**H0:** Mean sleep score is the **same** on rainy and non-rainy days.

**H1:** Mean sleep score is **different** on rainy vs non-rainy days.


In [ ]:
# Create a boolean for rain
df['is_rain'] = df['weather_condition'].str.contains('Rain', case=False)

rain_days = df.loc[df['is_rain'], 'sleep_score']
no_rain_days = df.loc[~df['is_rain'], 'sleep_score']

print('Number of rainy days:', len(rain_days))
print('Number of non-rainy days:', len(no_rain_days))

t_stat3, p_value3 = stats.ttest_ind(rain_days, no_rain_days, equal_var=False)
print('T-statistic:', t_stat3)
print('p-value:', p_value3)

if p_value3 < 0.05:
    print('Result: Reject H0 at 5% significance level. There is a significant difference.')
else:
    print('Result: Fail to reject H0 at 5% significance level. No significant difference found.')

## 9. Short Conclusions (To Be Written in Your Own Words)
- Summarize the main findings of your EDA (distribution of sleep, temperature, etc.).
- Interpret the p-values of the three hypothesis tests.
- Connect the results to your project question (e.g., *"Weather conditions seem to have limited impact on my sleep, but weekends slightly change my total sleep hours"*).

⚠️ **Not:** Bu kısmı mutlaka kendi cümlelerinle Türkçe veya İngilizce olarak yaz.
AI çıktısını direkt kopyalamayın; yorum ve cümleler size ait olsun.